# Post-hoc MTGFlow per SDE-Net diretto t+1…t+6

Questo notebook **non addestra** nulla. Legge `predictions.csv` della run prodotta da `notebooks/pvgis_sde_pipeline.ipynb`, rimuove le label salvate dal training e le ricalcola con un join puntuale sugli score MTGFlow alla coppia `(location, timestamp target)` di ogni canale.

Le label rispondono a una sola domanda: come ha previsto il modello i timestamp che MTGFlow considera anomali rispetto a quelli normali? Tutti gli output vanno in `EVALUATION_DIR`, separato dalla cartella del training. `event_group` non viene usato.

La decisione è `is_anomaly` del CSV MTGFlow (soglia per località del detector): nessun filtro qualità PVGIS e nessun ricalcolo top-K, così le label coincidono con quelle usate dal runner. Il notebook `stgan_pointwise_posthoc_sdenet.ipynb` applica lo stesso schema con STGAN.

## 1. Setup

In [ ]:
import os, subprocess, sys
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'physiq_pv').is_dir():
    for parent in REPO_ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', REPO_ROOT)

## 2. Run SDE-Net e score MTGFlow

`BASE_CONFIG` deve coincidere con `CONFIG` del notebook di training: serve a ritrovare la cartella della run e a passare `epochs`, `dropout` e `mc_samples` allo script di analisi. `SDE_MULTIHORIZON_PREDICTIONS` e `MTGFLOW_POSTHOC_ROOT` permettono di cambiare i percorsi senza modificare il notebook.

In [ ]:
FORECAST_HORIZONS = (1, 2, 3, 4, 5, 6)
PVGIS_DIR = pipe.PVGIS_DIR
DETECTOR = 'mtgflow'
DETECTOR_SEED = 15
DETECTOR_ROOT = Path('outputs/pvgis_mtgflow/downstream_dense') / f'seed_{DETECTOR_SEED}'
TEST_ANOMALY_SCORES = str(DETECTOR_ROOT / 'anomaly_scores.csv')

BASE_CONFIG = {
    **pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'horizon': 1,
    'forecast_horizons': ','.join(map(str, FORECAST_HORIZONS)),
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0,
    'n_sde_steps': 4, 'sigma_max': 0.5,
    'sde_sigma_initial': 0.01, 'sde_sigma_warmup_epochs': 30,
    'ood_noise_std': 2.0, 'mc_samples': 10, 'seed': 1,
    'ood_smoke_test': True, 'ood_smoke_max_samples': 2048,
    'anomaly_source': 'detector',
    'detector_regional_quantile': 0.975,
}
RUN_NAME = pipe.make_run_name(BASE_CONFIG)
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_MULTIHORIZON_PREDICTIONS',
    REPO_ROOT / pipe.make_out_dir(BASE_CONFIG) / 'predictions.csv',
)).resolve()
POSTHOC_ROOT = Path(os.environ.get(
    'MTGFLOW_POSTHOC_ROOT', REPO_ROOT / 'outputs' / 'sde_mtgflow_posthoc'
)).resolve()
# Una cartella per run SDE-Net e seed del detector: Run All la riutilizza.
EVALUATION_DIR = POSTHOC_ROOT / RUN_NAME / f'{DETECTOR}_seed{DETECTOR_SEED}'
MIN_MATCH_FRACTION = 0.90
FORCE_REBUILD_JOIN = False
RUN_ANALYSIS = True

checks = {
    'SDE-Net predictions': SDE_PREDICTIONS.is_file(),
    'MTGFlow test scores': Path(TEST_ANOMALY_SCORES).is_file(),
}
for name, present in checks.items():
    print(('OK     ' if present else 'MISSING') + '  ' + name)
if not checks['SDE-Net predictions']:
    raise FileNotFoundError(
        f'Eseguire prima notebooks/pvgis_sde_pipeline.ipynb: manca {SDE_PREDICTIONS}'
    )
if not checks['MTGFlow test scores']:
    raise FileNotFoundError(
        f'Eseguire prima notebooks/mtgflow_pvgis_workflow.ipynb: manca {TEST_ANOMALY_SCORES}'
    )
print('run:', RUN_NAME)
print('SDE direct predictions:', SDE_PREDICTIONS)
print('evaluation output:', EVALUATION_DIR)

## 3. Join puntuale con MTGFlow

Il join è inner su `(location, timestamp)` e condiviso dai sei orizzonti dello stesso target. Le righe SDE-Net senza score MTGFlow vengono escluse e contate nell'audit; sotto `MIN_MATCH_FRACTION` il notebook si ferma. Se input e impostazioni non cambiano, il join già scritto viene riutilizzato.

In [ ]:
JOIN_RESULT = build_pointwise_detector_evaluation(
    SDE_PREDICTIONS, TEST_ANOMALY_SCORES, EVALUATION_DIR,
    detector_name=DETECTOR,
    min_match_fraction=MIN_MATCH_FRACTION,
    allow_overwrite=True,
    reuse_existing=True,
    force_rebuild=FORCE_REBUILD_JOIN,
)
print('Join riutilizzato' if JOIN_RESULT['reused'] else 'Join rigenerato', EVALUATION_DIR)

AUDIT = json.loads((EVALUATION_DIR / 'evaluation_source.json').read_text(encoding='utf-8'))
display(pd.DataFrame([AUDIT])[[
    'detector', 'forecast_mode', 'horizons_hours', 'source_prediction_rows',
    'matched_rows', 'excluded_unmatched_rows', 'match_fraction',
    'original_detector_anomalies', 'normal_rows', 'rare_rows',
]].T.rename(columns={0: 'value'}))

PREDICTIONS = EVALUATION_DIR / 'predictions.csv'
joined = pd.read_csv(PREDICTIONS, usecols=lambda c: c in {
    'location', 'timestamp', 'horizon_hours', 'anomaly_group', 'event_group',
})
assert 'event_group' not in joined
assert joined['anomaly_group'].isin(['normal', 'rare_or_extreme']).all()
assert tuple(sorted(joined['horizon_hours'].unique())) == FORECAST_HORIZONS
assert not joined.duplicated(['location', 'timestamp', 'horizon_hours']).any()
display(joined.groupby(['horizon_hours', 'anomaly_group']).size().rename('rows').unstack(fill_value=0))
del joined

## 4. Serie temporale regionale MTGFlow — tutte le località

Come nel notebook post hoc STGAN, il pannello superiore mostra per ogni ora la **percentuale di località anomale fra quelle con score valido**. Il pannello inferiore conserva tutte le località con una riga per ID, una colonna per giorno e colore pari alla percentuale di ore valide classificate anomale (scala 0–100%).

Si includono tutte le ore con score, anche notturne, leggendo direttamente il CSV MTGFlow. La decisione è `anomaly_score >= threshold` salvata per quella località. Si escludono gli stessi dropout solari regionali e la recovery t+1 del post hoc STGAN; non si ricalcolano le soglie MTGFlow. Le ore mancanti o escluse rimangono interruzioni e le celle senza ore valide sono grigie.

Questa figura è indipendente dagli orizzonti SDE-Net: ciascuna coppia località/ora viene contata una sola volta. `REGIONAL_START` e `REGIONAL_END` selezionano un intervallo UTC con estremi inclusi; `None` mostra l'intero periodo degli score validi. I CSV esportano anche conteggi e copertura.

Per eseguire solo questa analisi bastano le sezioni 1–2. `PVGIS_2019_FILE` oppure `PVGIS_2019_PATH` può indicare il NetCDF per il filtro qualità.

In [ ]:
from IPython.display import Image, Markdown, display
from physiq_pv.reporting.stgan_regional import (
    load_mtgflow_regional_labels, build_detector_regional_overview,
)

REGIONAL_START = os.environ.get('MTGFLOW_REGIONAL_START') or None
REGIONAL_END = os.environ.get('MTGFLOW_REGIONAL_END') or None
REGIONAL_PVGIS_SOURCE = Path(os.environ.get(
    'PVGIS_2019_FILE', os.environ.get(
        'PVGIS_2019_PATH', str(Path(PVGIS_DIR) / 'piedmont_pvgis_2019.nc')
    ),
))
regional_labels = load_mtgflow_regional_labels(
    TEST_ANOMALY_SCORES, pvgis_quality_source=REGIONAL_PVGIS_SOURCE,
)
REGIONAL_PATHS = build_detector_regional_overview(
    regional_labels, EVALUATION_DIR / 'score_timeline', detector='mtgflow',
    start=REGIONAL_START, end=REGIONAL_END,
)
display(Image(filename=str(REGIONAL_PATHS['figure'])))
regional_hourly = pd.read_csv(REGIONAL_PATHS['hourly'])
display(Markdown('### Ore con la maggiore percentuale di località anomale'))
display(regional_hourly.nlargest(10, 'anomaly_share_pct'))
print('Figura e tabelle regionali:', {key: str(path) for key, path in REGIONAL_PATHS.items()})
del regional_labels

## 5. Post-hoc per `anomaly_group`

Le curve confrontano MAE e RMSE dei campioni normali e rari/anomali a t+1,…,t+6. Per t+1 e t+6 vengono inoltre mostrati box plot e istogrammi normalizzati dell'errore assoluto. Il grafico temporale mostra i sei canali diretti dello stesso modello e marca in rosso i target MTGFlow anomali.

In [ ]:
POSTHOC_LOCATION = None  # None: prima località disponibile
POSTHOC_START = None     # esempio: '2019-06-25'
POSTHOC_END = None       # esempio: '2019-07-02' (esclusivo)
POSTHOC_PATHS = pipe.build_direct_multihorizon_posthoc(
    EVALUATION_DIR, location=POSTHOC_LOCATION, start=POSTHOC_START, end=POSTHOC_END,
    detail_horizons=(1, 6),
)
metrics = pd.read_csv(POSTHOC_PATHS['metrics'])
display(metrics)
print({name: str(path) for name, path in POSTHOC_PATHS.items()})

In [ ]:
DETAILED_HORIZONS = (1, 6)
REQUIRED_FIGURE_PREFIXES = ('mae_', 'rmse_', 'nmpil_', 'picp_', 'clc_')
FULL_POSTHOC_FIGURES = {}
for horizon in DETAILED_HORIZONS:
    detail_out = EVALUATION_DIR / 'posthoc_by_horizon' / f't_plus_{horizon}'
    analysis_command = pipe.build_analysis_command(
        str(detail_out), BASE_CONFIG, predictions=str(PREDICTIONS),
        horizon_hours=horizon,
    )
    if RUN_ANALYSIS:
        subprocess.run(analysis_command, check=True, cwd=REPO_ROOT)
    elif not (detail_out / 'daytime_bin_anomaly_metrics.csv').is_file():
        raise FileNotFoundError(
            f'RUN_ANALYSIS=False ma manca il report per t+{horizon}: {detail_out}'
        )
    figures = pipe.build_posthoc_figures(
        str(detail_out), horizon_hours=horizon
    )
    missing = [prefix for prefix in REQUIRED_FIGURE_PREFIXES
               if not any(name.startswith(prefix) for name in figures)]
    if missing:
        raise RuntimeError(
            f'Suite post-hoc incompleta per t+{horizon}: prefissi mancanti {missing}'
        )
    FULL_POSTHOC_FIGURES[horizon] = figures
    print(f't+{horizon}: {len(figures)} figure complete per bin in {detail_out}')

In [ ]:
display(Image(filename=str(POSTHOC_PATHS['error_figure'])))
display(Image(filename=str(POSTHOC_PATHS['boxplot_figure'])))
display(Image(filename=str(POSTHOC_PATHS['histogram_figure'])))
display(Image(filename=str(POSTHOC_PATHS['prediction_figure'])))
for horizon, figures in FULL_POSTHOC_FIGURES.items():
    display(Markdown(f'## Suite paper-faithful t+{horizon}h'))
    for name, path in sorted(figures.items()):
        display(Markdown(f'**{name}**'))
        display(Image(filename=str(path)))